In [ ]:
import json
with open("D:\\GeoTKG\\cleandata\\tie\\train.json", "r") as f:
    examples=[json.loads(line) for line in f]

In [ ]:
counter = {"EVENT": 0, "TIME": 0, "DATE":0, "DURATION":0, "SET":0}

et_tot = 0
et_np_tot = 0

for example in examples:
    events = []
    times = []
    for inst in example['instances']:
        counter[inst['type']] += 1
        if inst['type'] == "EVENT":
            events.append(inst['id'])
        else:
            times.append(inst['id'])

    for et in example['event_times']:
        et_tot += 1

    et_np_tot += len(events)*len(times)
    

list(counter.values()), (et_np_tot-et_tot)/et_tot, et_tot, et_np_tot

In [ ]:
counter = {"BEFORE": 0, "AFTER": 0, "DURING":0, "CONTAINS":0, "OVERLAPS":0, "EQUALS":0, "IDENTITY":0}

for example in examples:
    for ee in example['ee_temprels']:
        counter[ee['rel']] += 1

counter

In [41]:
examples = []
with open("D:\\GeoTKG\\cleandata\\normalise\\eval.json", "r") as f:
    examples=[json.loads(line) for line in f]

import isodate

def gentext_to_iso8601(gentext: str):
    parsers = {
        isodate.parse_date:"DATE",
        isodate.parse_time:"TIME",
        isodate.parse_datetime:"TIME",
        isodate.parse_duration:"DURATION",
        isodate.parse_tzinfo:"SET",
    }

    for parser in parsers:
        try:
            out = parser(gentext)
            type_ = parsers[parser]
            return out, type_
        except Exception:
            continue

    # If none of the parsers worked
    print(f"UNREC: {gentext}")
    return None

deecode = []
truth = []
types = []
rand = 10
for i,ex in enumerate(examples):
    ti_type = ex['input_text'].split(" ")[3]
    outer = '1999-09-09' if i%rand==0 else ex["output_text"]
    deecode.append(outer)
    truth.append(ex["output_text"])
    types.append(ti_type)


In [47]:
import datetime as _dt
from typing import List, Tuple, Optional

# assumes you already defined this (your cleaner loop-based version)
# from your_module import gentext_to_iso8601

def _cmp_date_components(gold, pred) -> bool:
    """Any of year/month/day matches."""
    g = {"y": getattr(gold, "year", None), "m": getattr(gold, "month", None), "d": getattr(gold, "day", None)}
    p = {"y": getattr(pred, "year", None), "m": getattr(pred, "month", None), "d": getattr(pred, "day", None)}
    return any(g[k] is not None and g[k] == p[k] for k in ("y", "m", "d"))

def _cmp_time_components(gold, pred) -> bool:
    """Any of hour/minute/second matches (ignores microseconds)."""
    g = {"h": getattr(gold, "hour", None), "m": getattr(gold, "minute", None), "s": getattr(gold, "second", None)}
    p = {"h": getattr(pred, "hour", None), "m": getattr(pred, "minute", None), "s": getattr(pred, "second", None)}
    return any(g[k] is not None and g[k] == p[k] for k in ("h", "m", "s"))

def _cmp_datetime_components(gold, pred) -> bool:
    """Any date *or* time component matches."""
    date_ok = _cmp_date_components(gold, pred)
    time_ok = _cmp_time_components(gold, pred)
    return date_ok or time_ok

def _total_seconds(x) -> Optional[float]:
    # isodate durations often become datetime.timedelta
    if isinstance(x, _dt.timedelta):
        return x.total_seconds()
    return None

def _cmp_duration_any_component(gold, pred) -> bool:
    """
    Mark correct if total seconds equal (most practical),
    OR if both encode at least one matching component (days/hours/minutes/seconds) when derivable.
    """
    gs = _total_seconds(gold)
    ps = _total_seconds(pred)
    if gs is not None and ps is not None:
        return abs(gs - ps) < 1e-6

    # Fallback: try to infer rough components if timedelta-like but not precise
    # (Most libraries give timedelta; if not, we can’t safely decompose—return False.)
    return False

def _normalize_set_string(s: str) -> str:
    """
    Very light 'SET' normalization:
    - split common separators, strip whitespace, sort tokens, rejoin.
    Adjust to your dataset’s SET format.
    """
    for sep in [",", ";", "|"]:
        s = s.replace(sep, " ")
    toks = [t for t in s.split() if t]
    toks.sort()
    return " ".join(toks).lower()

def _cmp_set_relaxed(gold_str: str, pred_str: str) -> bool:
    """Any overlap in normalized token sets qualifies as relaxed-correct."""
    g = set(_normalize_set_string(gold_str).split())
    p = set(_normalize_set_string(pred_str).split())
    return len(g & p) > 0

def relaxed_correct_single(gold_text: str, pred_text: str) -> bool:
    """
    Strict equality first; if not equal, apply relaxed rule per ti_type.
    ti_type ∈ {"DATE","TIME","DATETIME","DURATION","SET"} (case-insensitive).
    """
    # Strict exact match first (you can move strict to your main metric if preferred)
    if pred_text == gold_text:
        return True

    g_parsed,t = gentext_to_iso8601(gold_text)
    p_parsed,t = gentext_to_iso8601(pred_text)

    # If parsing fails for either side, fall back to string-based relaxed checks for SET,
    # otherwise we can’t relax-match.
    if g_parsed is None or p_parsed is None:
        if t == "SET":
            return _cmp_set_relaxed(gold_text, pred_text)
        return False

    if t == "DATE":
        # Any of year/month/day matches
        return _cmp_date_components(g_parsed, p_parsed)

    if t == "TIME":
        # Any of hour/minute/second matches
        return _cmp_time_components(g_parsed, p_parsed)

    if t in ("DATETIME", "DATE-TIME", "DATE_TIME"):
        # Any date OR time component matches
        return _cmp_datetime_components(g_parsed, p_parsed)

    if t == "DURATION":
        # Durations equal in total seconds (or fallback logic)
        return _cmp_duration_any_component(g_parsed, p_parsed)

    if t == "SET":
        # Any overlapping value in normalized token sets
        return _cmp_set_relaxed(gold_text, pred_text)

    # Unknown type → no relaxed match
    return False

def relaxed_accuracy(
    decoded_labels: List[str],
    decoded_preds: List[str],
) -> float:
    """
    Vectorized relaxed accuracy:
    - If strict match: correct
    - Else: relaxed_correct_single()
    """
    assert len(decoded_labels) == len(decoded_preds)
    correct = 0
    for y, p in zip(decoded_labels, decoded_preds):
        if p == y or relaxed_correct_single(y, p):
            correct += 1
        elif p!="1999-09-09":
            print(y, p)
    return correct / max(1, len(decoded_labels))


In [48]:
relaxed_accuracy(truth,deecode)

0.9026128266033254

In [49]:
for t in truth:
    if t[-3:] == "REF":
        print(t)

PRESENT_REF
PAST_REF
FUTURE_REF
PRESENT_REF
PRESENT_REF
FUTURE_REF
PAST_REF
FUTURE_REF
PRESENT_REF
PAST_REF
PRESENT_REF
PAST_REF
PRESENT_REF
PAST_REF
FUTURE_REF
PRESENT_REF
PRESENT_REF
PRESENT_REF
PRESENT_REF
PAST_REF
PRESENT_REF
PRESENT_REF
PRESENT_REF
PRESENT_REF
PRESENT_REF
PRESENT_REF
PAST_REF
PRESENT_REF
PRESENT_REF
PRESENT_REF
FUTURE_REF
PRESENT_REF
PRESENT_REF
PRESENT_REF
FUTURE_REF
PRESENT_REF
PAST_REF
PRESENT_REF
PRESENT_REF


TypeError: 'NoneType' object is not subscriptable

In [10]:
import json
examples = []
with open("D:\\GeoTKG\\cleandata\\normalise\\train.json", "r") as f:
    examples=[json.loads(line) for line in f]

maxi = ""
for ex in examples:
    maxi = ex['input_text'] if len(ex['input_text'])>len(maxi) else maxi

In [11]:
maxi

'DCT: 2005-11-29 \nTYPE: DATE \nTEXT: youthsShare prices end   lower in Hong Kong -- Nov. 29 China vows   to protect   migrant workers against HIV / AIDSChinese top legislator meets   Mongolian presidentMore needy college students benefit   from KFC foundationUSAID official : China becomes   major contributor to int\'l aid for educationChinese top political advisor meets   Mongolian presidentAncient limestone cave suffers   damage in Inner MongoliaChina Gas may announce   new potential strategic investorIndian official : India to learn   from China for educational developmentWeather information for major Asia - Pacific cities -- Nov. 29 Hong Kong gold price ends higher -- Nov. 29 China , Mongolia issue   joint statement , pledging   closer tiesChina to reduce   slick impact   on downstream Russia : FM spokesmanNW China \'s Xinjiang boasts   thousands of scenic spotsWTO MC6 appoints host   HK constitutional development forumHang Seng China Enterprises Index closes   lower -- Nov. 29 Pol

In [12]:
len(maxi)

1315